# prep_traduccion

Preprocesamiento (no es un experimento: solo genera datos).

Traduce al inglés las partes de `Description` que están en **malayo** o **chino**, con el modelo local
`Helsinki-NLP/opus-mt-mul-en` (MarianMT). No usa servicios externos.

Se separa cada descripción en frases, y solo se traducen las frases que tienen caracteres chinos o
palabras típicas del malayo. Las frases en inglés quedan como están.

Salida: `train.parquet` y `test.parquet` con índice `PetID` y la columna `Description_en`, con el texto
completo para todas las mascotas (traducido o igual al original), más `traducida` (0/1).

In [ ]:
import os, re, time
import numpy as np
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer

torch.manual_seed(42)
MODELO = "Helsinki-NLP/opus-mt-mul-en"
tok = MarianTokenizer.from_pretrained(MODELO)
mod = MarianMTModel.from_pretrained(MODELO).eval()

MALAYO = set("""dan yang untuk anjing kucing sangat boleh ini itu tidak dengan saya anak comel manja sihat sudah belum
ada mahu nak dia kami kita akan atau juga lagi kalau jika sebab kerana pada dari ke di oleh telah dapat jaga
rumah makan suka baik cantik sila hubungi ekor jantan betina bulan tahun umur vaksin mandul percuma adopt""".split())
CJK = re.compile(r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af]")

def necesita_traduccion(frase):
    if CJK.search(frase):
        return True
    palabras = re.findall(r"[a-z]+", frase.lower())
    n_ms = sum(p in MALAYO for p in palabras)
    return n_ms >= 2 and n_ms / max(len(palabras), 1) >= 0.15

def frases(texto):
    return [f for f in re.split(r"(?<=[.!?。！？\n])\s*", texto) if f.strip()]

def traducir(lista, lote=16):
    salida = []
    for a in range(0, len(lista), lote):
        enc = tok(lista[a:a + lote], return_tensors="pt", padding=True, truncation=True, max_length=256)
        with torch.no_grad():
            gen = mod.generate(**enc, max_new_tokens=256)
        salida += tok.batch_decode(gen, skip_special_tokens=True)
    return salida

def procesar(df):
    textos = df["Description"].fillna("").astype(str)
    partes = {pid: frases(t) for pid, t in textos.items()}
    pendientes = sorted({f for fs in partes.values() for f in fs if necesita_traduccion(f)})
    t0 = time.time()
    hechas = dict(zip(pendientes, traducir(pendientes)))
    print(f"frases traducidas: {len(pendientes)} en {time.time() - t0:.0f} s")
    nuevo, marca = {}, {}
    for pid, fs in partes.items():
        cambio = [hechas.get(f, f) for f in fs]
        marca[pid] = int(any(f in hechas for f in fs))
        nuevo[pid] = " ".join(cambio) if marca[pid] else textos[pid]
    return pd.DataFrame({"Description_en": pd.Series(nuevo), "traducida": pd.Series(marca)}).loc[df.index]

In [ ]:
RAIZ = "/kaggle/input" if os.path.exists("/kaggle/input") else "../input"
import glob
COMP = os.path.dirname(os.path.dirname(glob.glob(f"{RAIZ}/**/train/train.csv", recursive=True)[0]))
SALIDA = "." if os.path.exists("/kaggle") else "../input/lab2-traduccion"
os.makedirs(SALIDA, exist_ok=True)
for conjunto in ["train", "test"]:
    df = pd.read_csv(f"{COMP}/{conjunto}/{conjunto}.csv", index_col="PetID", usecols=["PetID", "Description"])
    res = procesar(df)
    res.to_parquet(os.path.join(SALIDA, f"{conjunto}.parquet"))
    print(conjunto, res.shape, "| descripciones con algo traducido:", res["traducida"].sum())

In [ ]:
# Ejemplos (antes -> después)
ej = pd.read_csv(f"{COMP}/train/train.csv", index_col="PetID", usecols=["PetID", "Description"]).join(
    pd.read_parquet(os.path.join(SALIDA, "train.parquet")))
for pid, fila in ej[ej["traducida"] == 1].head(6).iterrows():
    print(pid, "|", fila["Description"][:200].encode("ascii", "replace").decode(), "\n   ->", fila["Description_en"][:200], "\n")